# BioChannel Notebook 2 — Information Loss Analyzer

**Build an information-theoretic channel from time-series gene expression.**

This notebook is part of the **BioChannel** Kaggle hackathon project. It is designed to be run inside Kaggle Notebooks and then reused by the Streamlit dashboard.

## How to use this notebook in BioChannel

1. Attach the relevant Kaggle dataset using **Add Data** in the Kaggle notebook sidebar.
2. Run the notebook end-to-end.
3. Save processed outputs into `/kaggle/working/processed/`.
4. Copy the exported CSV/JSON files into the BioChannel Streamlit app under `data/processed/`.
5. Reuse the helper functions in the app modules: `data_loader.py`, `preprocessing.py`, `simulation.py`, `info_theory.py`, and `prediction.py`.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import mutual_info_score
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error

np.random.seed(42)


In [ ]:
from pathlib import Path

INPUT_DIR = Path('/kaggle/input')
WORKING_DIR = Path('/kaggle/working')
PROCESSED_DIR = WORKING_DIR / 'processed'
PROCESSED_DIR.mkdir(exist_ok=True)

print('Input datasets available:')
for p in INPUT_DIR.glob('*'):
    print(' -', p)

## Dataset

Recommended dataset:

- Time-series gene expression: `soham1024/gene-expression-time-series`
- Alternative: NCBI GEO signaling time-course datasets

Science mapping:

`X(t) = stimulus/input signal`

`Y(t) = pathway or gene-expression response`

BioChannel estimates entropy, mutual information, signal fidelity, and information loss.

In [ ]:
csv_files = list(INPUT_DIR.rglob('*.csv'))
for f in csv_files[:20]: print(f)

In [ ]:
def load_time_series_or_demo(csv_files):
    for f in csv_files:
        try:
            df = pd.read_csv(f)
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            if len(numeric_cols) >= 3 and len(df) >= 20:
                print('Using:', f)
                return df
        except Exception:
            pass
    print('No suitable dataset found. Creating demo time-series data.')
    time = np.linspace(0, 12, 120)
    input_signal = np.sin(time) + 0.35*np.sin(2.3*time)
    response = np.roll(input_signal, 5) + np.random.normal(0, 0.25, len(time))
    return pd.DataFrame({'time': time, 'input_signal': input_signal, 'pathway_response': response})

df = load_time_series_or_demo(csv_files)
df.head()

In [ ]:
numeric = df.select_dtypes(include=[np.number]).copy().fillna(0)
cols = list(numeric.columns)

# Try to infer time column.
time_col = next((c for c in cols if 'time' in c.lower()), cols[0])
time = numeric[time_col].values

# If explicit signal/response columns exist, use them. Otherwise synthesize input and use average expression response.
signal_col = next((c for c in cols if 'signal' in c.lower() or 'stim' in c.lower()), None)
response_col = next((c for c in cols if 'response' in c.lower() or 'pathway' in c.lower()), None)

if signal_col is None:
    signal = np.sin(np.linspace(0, 3*np.pi, len(numeric)))
else:
    signal = numeric[signal_col].values

if response_col is None:
    response_cols = [c for c in cols if c != time_col and c != signal_col]
    response = numeric[response_cols].mean(axis=1).values
else:
    response = numeric[response_col].values

channel_df = pd.DataFrame({'time': time, 'input_signal': signal, 'output_response': response})
channel_df.to_csv(PROCESSED_DIR / 'info_loss_channel_timeseries.csv', index=False)
channel_df.head()

In [ ]:
def discretize(values, bins=10):
    return pd.cut(values, bins=bins, labels=False, duplicates='drop').astype(int)

def entropy_bits(values, bins=10):
    labels = discretize(values, bins)
    counts = np.bincount(labels)
    probs = counts[counts > 0] / counts.sum()
    return float(-np.sum(probs * np.log2(probs)))

def information_metrics(x, y, bins=10):
    xd = discretize(x, bins)
    yd = discretize(y, bins)
    h_x = entropy_bits(x, bins)
    h_y = entropy_bits(y, bins)
    mi = mutual_info_score(xd, yd) / np.log(2)
    fidelity = float(np.clip(mi / h_x, 0, 1)) if h_x > 0 else 0
    return {
        'input_entropy_bits': h_x,
        'output_entropy_bits': h_y,
        'mutual_information_bits': float(mi),
        'signal_fidelity': fidelity,
        'information_loss': 1 - fidelity,
    }

metrics = information_metrics(channel_df['input_signal'], channel_df['output_response'])
with open(PROCESSED_DIR / 'info_loss_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
metrics

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(channel_df['time'], channel_df['input_signal'], label='Input signal')
plt.plot(channel_df['time'], channel_df['output_response'], label='Output response')
plt.title('BioChannel: Input Signal vs Output Response')
plt.xlabel('Time')
plt.ylabel('Signal / response')
plt.legend()
plt.show()

plt.figure(figsize=(5,4))
plt.bar(['Signal fidelity','Information loss'], [metrics['signal_fidelity'], metrics['information_loss']])
plt.ylim(0,1)
plt.title('Information Preservation')
plt.show()

## Streamlit integration

Use:

- `info_loss_channel_timeseries.csv` → input/output time-series plot
- `info_loss_metrics.json` → dashboard metric cards

Gemma prompt should explain whether the selected condition resembles reliable signaling, noisy ambiguity, crosstalk, feedback instability, or loss of pathway fidelity.